## Setup
    - Run an dokcer image of mysql server


In [2]:
!pip install mysql-connector-python pandas tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 735.8 kB/s  0:00:29m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [mysql-connector-python]-connector-python]

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import mysql.connector
from mysql.connector import Error
import pandas as pd
from tabulate import tabulate

### Databse creation and data population

In [5]:
def get_connection(database=None):
    return mysql.connector.connect(
        host="localhost",
        user="root",
        password="root123",   # <-- change this
        database="movies"
    )

conn = get_connection()
cursor = conn.cursor()

In [6]:
# cursor.execute("CREATE DATABASE IF NOT EXISTS movies")
# cursor.close()
# conn.close()

In [8]:
conn = get_connection()
cursor = conn.cursor()

In [9]:
# Create a cathroy table
cursor.execute("""
                CREATE TABLE IF NOT EXISTS category (
                    category_id INT AUTO_INCREMENT PRIMARY KEY,
                    name        VARCHAR(50) NOT NULL
                )
""")
conn.commit()

In [12]:
# create film table

cursor.execute("""
CREATE TABLE IF NOT EXISTS film (
        film_id      INT AUTO_INCREMENT PRIMARY KEY,
        title        VARCHAR(255) NOT NULL,
        description  TEXT,
        release_year YEAR,
        length       INT,
        rating       ENUM('G','PG','PG-13','R','NC-17')
    )
""")
conn.commit()

In [13]:
# create film_catgeory table

cursor.execute("""

CREATE TABLE IF NOT EXISTS film_category (
        film_id     INT NOT NULL,
        category_id INT NOT NULL,
        PRIMARY KEY (film_id, category_id),
        FOREIGN KEY (film_id) REFERENCES film(film_id),
        FOREIGN KEY (category_id) REFERENCES category(category_id)
    )

""")
conn.commit()

In [14]:
# create staff table

cursor.execute("""

 CREATE TABLE IF NOT EXISTS staff (
        staff_id   INT AUTO_INCREMENT PRIMARY KEY,
        first_name VARCHAR(50),
        last_name  VARCHAR(50),
        email      VARCHAR(100),
        store_id   INT
    )

""")
conn.commit()

In [16]:
# create store table

cursor.execute("""

 CREATE TABLE IF NOT EXISTS store (
        store_id         INT AUTO_INCREMENT PRIMARY KEY,
        manager_staff_id INT,
        FOREIGN KEY (manager_staff_id) REFERENCES staff(staff_id)
    )

""")
conn.commit()

### Inserting Sample data

In [17]:
# Categories
categories = ['Action', 'Comedy', 'Drama', 'Horror', 'Animation', 'Documentary']
cursor.executemany(
    "INSERT INTO category (name) VALUES (%s)",
    [(c,) for c in categories]
)

In [18]:
# Films
films = [
    ('Fast Pursuit', 'A high speed chase thriller', 2015, 120, 'PG-13'),
    ('Laugh Out Loud', 'A comedy about friendship', 2018, 95, 'PG'),
    ('Silent Echoes', 'A dramatic tale of loss', 2012, 150, 'R'),
    ('Haunted Manor', 'A horror story in an old house', 2020, 110, 'R'),
    ('Toy Kingdom', 'An animated adventure for kids', 2019, 90, 'G'),
    ('Deep Space Voyage', 'Sci-fi thriller in space', 2021, 160, 'PG-13'),
    ('City of Shadows', 'Crime drama', 2016, 130, 'R'),
    ('Funny Bones', 'Slapstick comedy', 2014, 85, 'PG'),
    ('The Last Stand', 'Action packed war film', 2017, 140, 'PG-13'),
    ('Whispering Woods', 'Family drama', 2013, 100, 'PG'),
    ('Nightmare Street', 'Horror sequel', 2022, 105, 'NC-17'),
    ('Animated Dreams', 'Animation fantasy', 2020, 88, 'G'),
]
cursor.executemany(
    "INSERT INTO film (title, description, release_year, length, rating) "
    "VALUES (%s, %s, %s, %s, %s)",
    films
)

In [19]:
# film_category mapping (film_id, category_id) — matches insert order (1-12)
film_category_map = [
    (1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 1),
    (7, 3), (8, 2), (9, 1), (10, 3), (11, 4), (12, 5)
]
cursor.executemany(
    "INSERT INTO film_category (film_id, category_id) VALUES (%s, %s)",
    film_category_map
)

In [20]:
# Staff
staff = [
    ('John', 'Smith', 'john.smith@example.com', 1),
    ('Emily', 'Davis', 'emily.davis@example.com', 2),
]
cursor.executemany(
    "INSERT INTO staff (first_name, last_name, email, store_id) VALUES (%s, %s, %s, %s)",
    staff
)

In [21]:
# Store
stores = [(1,), (2,)]
cursor.executemany(
    "INSERT INTO store (manager_staff_id) VALUES (%s)",
    stores
)

In [29]:
## helper function to run a query and visualise it as pandas dataframe
def run_query(query, conn=conn):

    df = pd.read_sql(query, conn)
    # print(tabulate(df, headers='keys', tablefmt='psql', showindex=False))
    return df

## Question 1: Basic Data Retrieval
    - Write an SQL query to display all columns from the `film` table.

In [30]:
query1 = "SELECT * FROM film;"
df1 = run_query(query1, conn)
df1

/tmp/ipykernel_5651/4186291813.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,film_id,title,description,release_year,length,rating
0,1,Fast Pursuit,A high speed chase thriller,2015,120,PG-13
1,2,Laugh Out Loud,A comedy about friendship,2018,95,PG
2,3,Silent Echoes,A dramatic tale of loss,2012,150,R
3,4,Haunted Manor,A horror story in an old house,2020,110,R
4,5,Toy Kingdom,An animated adventure for kids,2019,90,G
5,6,Deep Space Voyage,Sci-fi thriller in space,2021,160,PG-13
6,7,City of Shadows,Crime drama,2016,130,R
7,8,Funny Bones,Slapstick comedy,2014,85,PG
8,9,The Last Stand,Action packed war film,2017,140,PG-13
9,10,Whispering Woods,Family drama,2013,100,PG


## Question 2: Filtering Records
    - Retrieve the title and rating of all films with rating `PG` or `PG-13`.

In [31]:
query2 = "SELECT * FROM film WHERE rating == 'PG' or rating = 'PG-13';"
df2 = run_query(query1, conn)
df2

/tmp/ipykernel_5651/4186291813.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,film_id,title,description,release_year,length,rating
0,1,Fast Pursuit,A high speed chase thriller,2015,120,PG-13
1,2,Laugh Out Loud,A comedy about friendship,2018,95,PG
2,3,Silent Echoes,A dramatic tale of loss,2012,150,R
3,4,Haunted Manor,A horror story in an old house,2020,110,R
4,5,Toy Kingdom,An animated adventure for kids,2019,90,G
5,6,Deep Space Voyage,Sci-fi thriller in space,2021,160,PG-13
6,7,City of Shadows,Crime drama,2016,130,R
7,8,Funny Bones,Slapstick comedy,2014,85,PG
8,9,The Last Stand,Action packed war film,2017,140,PG-13
9,10,Whispering Woods,Family drama,2013,100,PG


## Question 3: Sorting and Limiting Results
List all films ordered by length (descending). Display only the top two longest films.

In [33]:
query3 = "SELECT * FROM film ORDER BY length DESC LIMIT 2"
df3 = run_query(query3, conn)
df3

/tmp/ipykernel_5651/4186291813.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,film_id,title,description,release_year,length,rating
0,6,Deep Space Voyage,Sci-fi thriller in space,2021,160,PG-13
1,3,Silent Echoes,A dramatic tale of loss,2012,150,R


## Question 4: Aggregate Functions
    - Calculate the average length of all films. Rename the column as `average_film_length`.

In [35]:
query4 = """
    SELECT AVG(length) AS average_film_length FROM film
"""

df4 = run_query(query4, conn)
df4

/tmp/ipykernel_5651/4186291813.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,average_film_length
0,114.4167


## Question 5: Joining Tables
Display each film title along with its category name.

In [39]:
query5 = """
    SELECT f.title, c.name AS category_name
    FROM film f
    JOIN film_category fc ON f.film_id = fc.film_id
    JOIN category c ON fc.category_id = c.category_id;
"""
df5 = run_query(query5, conn)
df5

/tmp/ipykernel_5651/4186291813.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,title,category_name
0,Fast Pursuit,Action
1,Deep Space Voyage,Action
2,The Last Stand,Action
3,Laugh Out Loud,Comedy
4,Funny Bones,Comedy
5,Silent Echoes,Drama
6,City of Shadows,Drama
7,Whispering Woods,Drama
8,Haunted Manor,Horror
9,Nightmare Street,Horror


## Question 6: GROUP BY and COUNT
Show the number of films in each category, sorted from highest to lowest count.

In [40]:
query6 = """
    SELECT c.name AS category_name, COUNT(fc.film_id) AS film_count
    FROM category c
    JOIN film_category fc ON c.category_id = fc.category_id
    GROUP BY c.category_id, c.name
    ORDER BY film_count DESC;
"""
df6 = run_query(query6, conn)
df6

/tmp/ipykernel_5651/4186291813.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,category_name,film_count
0,Action,3
1,Drama,3
2,Comedy,2
3,Horror,2
4,Animation,2


## Question 7: Handling Duplicate Data
Display distinct category names only.

In [41]:
query7 = """
    SELECT DISTINCT name FROM category;
"""
df7 = run_query(query7, conn)
df7

/tmp/ipykernel_5651/4186291813.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,name
0,Action
1,Comedy
2,Drama
3,Horror
4,Animation
5,Documentary


## Question 8: Conditional Logic
Display film title and classify films as `Long Film` or `Short Film`.

In [42]:
query8 = """
    SELECT title,
           CASE
               WHEN length > 120 THEN 'Long Film'
               ELSE 'Short Film'
           END AS film_length_category
    FROM film;
"""
df8 = run_query(query8, conn)
df8

/tmp/ipykernel_5651/4186291813.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,title,film_length_category
0,Fast Pursuit,Short Film
1,Laugh Out Loud,Short Film
2,Silent Echoes,Long Film
3,Haunted Manor,Short Film
4,Toy Kingdom,Short Film
5,Deep Space Voyage,Long Film
6,City of Shadows,Long Film
7,Funny Bones,Short Film
8,The Last Stand,Long Film
9,Whispering Woods,Short Film


## Question 9: Multi-Table Query
Display Store ID, Manager’s full name, and Manager’s email.

In [43]:
query9 = """
    SELECT s.store_id,
           CONCAT(st.first_name, ' ', st.last_name) AS manager_full_name,
           st.email AS manager_email
    FROM store s
    JOIN staff st ON s.manager_staff_id = st.staff_id;
"""
df9 = run_query(query9, conn)
df9

/tmp/ipykernel_5651/4186291813.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,store_id,manager_full_name,manager_email
0,1,John Smith,john.smith@example.com
1,2,Emily Davis,emily.davis@example.com


## Question 10: Data Validation Query
Identify categories that have no films assigned to them.

In [44]:
query10 = """
    SELECT c.name AS category_name
    FROM category c
    LEFT JOIN film_category fc ON c.category_id = fc.category_id
    WHERE fc.film_id IS NULL;
"""
df10 = run_query(query10, conn)
df10

/tmp/ipykernel_5651/4186291813.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,category_name
0,Documentary


## Challenge 1 (Optional)
Retrieve all film data and display it using `tabulate`.

In [46]:
## helper function to run a query and visualise it as pandas dataframe
def run_query(query, conn=conn):

    df = pd.read_sql(query, conn)
    return tabulate(df, headers='keys', tablefmt='psql', showindex=False)

In [47]:
query11 = """
    SELECT * FROM film
"""

df11 = run_query(query11, conn)
df1

/tmp/ipykernel_5651/200643817.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,film_id,title,description,release_year,length,rating
0,1,Fast Pursuit,A high speed chase thriller,2015,120,PG-13
1,2,Laugh Out Loud,A comedy about friendship,2018,95,PG
2,3,Silent Echoes,A dramatic tale of loss,2012,150,R
3,4,Haunted Manor,A horror story in an old house,2020,110,R
4,5,Toy Kingdom,An animated adventure for kids,2019,90,G
5,6,Deep Space Voyage,Sci-fi thriller in space,2021,160,PG-13
6,7,City of Shadows,Crime drama,2016,130,R
7,8,Funny Bones,Slapstick comedy,2014,85,PG
8,9,The Last Stand,Action packed war film,2017,140,PG-13
9,10,Whispering Woods,Family drama,2013,100,PG
